# AIMNet2: exposing the electrostatic coupling head

Load an AIMNet2 calculator, inspect its PyTorch model, and add the Coulomb coupling term as a separate output. This term is useful when studying charge response: its derivative with respect to atomic charges is an electrostatic potential contribution. This notebook exposes the energy term; it does not calculate a polarizability.

Use the demo's `aimnet2` kernel. The first run may download model weights. Run cells in order from a fresh kernel.

In [ ]:
import os
import sys

# On macOS, avoid mixed-architecture TorchInductor caches. Set before importing torch.
if sys.platform == 'darwin':
    os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')

import copy
import torch
from torch import nn
from ase.build import molecule
from aimnet.calculators import AIMNet2Calculator

calculator = AIMNet2Calculator('aimnet2', device='cpu')
model = calculator.model
print('Calculator model type:', type(model))
print(model)

## Find the coupling module

Some AIMNet2 models put `lrcoulomb` in `model.outputs`. Current models may attach the long range Coulomb module to the calculator and put a short range correction in `model.outputs`. Select the module actually used by the loaded calculator, so its parameters and cutoff match that model.

In [ ]:
if not isinstance(model.outputs, nn.ModuleDict):
    raise TypeError('This example expects model.outputs to be a torch.nn.ModuleDict')

print('Model output modules:', list(model.outputs))
embedded = model.outputs['lrcoulomb'] if 'lrcoulomb' in model.outputs else None
external = getattr(calculator, 'external_coulomb', None)

if embedded is not None:
    coupling_head = embedded
    source = "model.outputs['lrcoulomb']"
elif external is not None:
    coupling_head = external
    source = 'calculator.external_coulomb'
else:
    raise RuntimeError('No Coulomb coupling module found on this AIMNet2 calculator')

print('Coupling module source:', source)
print('Coupling module:', coupling_head)
print('Input key:', coupling_head.key_in, 'original output key:', coupling_head.key_out)
if 'srcoulomb' in model.outputs:
    print('Short range correction:', model.outputs['srcoulomb'])

## Baseline calculation

Use the calculator's dictionary interface to retrieve model outputs. The ASE wrapper exposes a fixed set of properties and will not return this additional key.

In [ ]:
water = molecule('H2O')
inputs = {'coord': water.positions, 'numbers': water.numbers, 'charge': 0.0}
baseline = calculator.eval(inputs)
print('Baseline energy (eV):', baseline['energy'].detach().cpu().numpy())
print('Atomic charges (e):', baseline['charges'].detach().cpu().numpy())

## Register a separate output head

Copy the loaded coupling module and change the copy's output key. The original energy contribution remains in place. AIMNet2 calls output modules in insertion order, so the new head runs after the existing heads. The calculator filters returned keys, so add the new key to its output list too.

In [ ]:
output_key = 'electrostatic_energy'
head_name = 'electrostatic_probe'
if head_name in model.outputs or output_key in calculator.keys_out:
    raise RuntimeError('Probe already registered; restart the kernel before rerunning this cell')

probe = copy.deepcopy(coupling_head)
probe.key_out = output_key
model.outputs[head_name] = probe
calculator.keys_out = [*calculator.keys_out, output_key]

print('Registered:', head_name, '->', model.outputs[head_name])
print('Returned keys:', calculator.keys_out)

In [ ]:
result = calculator.eval(inputs)
print('Returned keys:', list(result))
print('Total energy (eV):', result['energy'].detach().cpu().numpy())
print('Electrostatic coupling energy (eV):', result[output_key].detach().cpu().numpy())

torch.testing.assert_close(result['energy'], baseline['energy'])
torch.testing.assert_close(result['charges'], baseline['charges'])
assert result[output_key].numel() == result['energy'].numel()
print('Verified: total energy and atomic charges are unchanged.')

The separate value is the selected Coulomb module's energy contribution, in eV per structure. With the current default model, this is the external full Coulomb term, while `srcoulomb` supplies a short range subtraction in the total energy. With an embedded `lrcoulomb` model, it is that embedded head's term. Access the added output on later calls with `calculator.eval(inputs)['electrostatic_energy']`.